In [1]:
import tensorflow as tf
import pandas as pd
import numpy as np
import math

from sqlalchemy import create_engine
from tensorflow.keras import layers
from tensorflow.keras import regularizers

In [29]:
dbConnectionString = "sqlite:///baseball_info.db"

engine = create_engine(dbConnectionString)

In [30]:
averageHittingQuery = "SELECT playerId,AVG(plateAppearances),AVG(atBats),AVG(runs),AVG(hits),AVG(singles),AVG(doubles),AVG(triples),AVG(homeRuns),AVG(rbis),AVG(sacHits),AVG(sacFlies),AVG(hitByPitch),AVG(walks),AVG(intentionalWalks),AVG(strikeOuts),AVG(stolenBases),AVG(caughtStealing),AVG(groundedIntoDoublePlays),AVG(catcherInterference),AVG(battingAverage),AVG(onBasePercentage),AVG(sluggingPercentage),AVG(onBasePlusSlugging) FROM SeasonStatsHitting WHERE season between 2022 and 2024 GROUP BY playerId"

df = pd.read_sql(averageHittingQuery, engine)


In [31]:
lastSeasonHittingStatsQuery = "SELECT playerId,rbis,homeRuns,runs,onBasePercentage,stolenBases FROM SeasonStatsHitting WHERE season=2025"

dfLabels = pd.read_sql(lastSeasonHittingStatsQuery, engine)

In [32]:
dfFilteredTrain  = df      [df      ['playerId'].isin(dfLabels       ['playerId'])]
dfFilteredLabels = dfLabels[dfLabels['playerId'].isin(dfFilteredTrain['playerId'])]

In [33]:
print(len(dfFilteredTrain))
print(len(dfFilteredLabels))

597
597


In [34]:
dfFilteredTrainSorted  = dfFilteredTrain .sort_values(by='playerId')
dfFilteredLabelsSorted = dfFilteredLabels.sort_values(by='playerId')

dfFilteredTrainSorted  = dfFilteredTrainSorted .set_index("playerId").reset_index()
dfFilteredLabelsSorted = dfFilteredLabelsSorted.set_index("playerId").reset_index()

In [35]:
dfFilteredTrainSorted.head()

,playerId,AVG(plateAppearances),AVG(atBats),AVG(runs),AVG(hits),AVG(singles),AVG(doubles),AVG(triples),AVG(homeRuns),AVG(rbis),...,AVG(intentionalWalks),AVG(strikeOuts),AVG(stolenBases),AVG(caughtStealing),AVG(groundedIntoDoublePlays),AVG(catcherInterference),AVG(battingAverage),AVG(onBasePercentage),AVG(sluggingPercentage),AVG(onBasePlusSlugging)
0,445276,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
1,455117,311.000000,280.333333,27.333333,49.666667,29.333333,9.000000,0.000000,11.333333,30.666667,...,0.000000,102.000000,0.000000,0.000000,7.000000,0.0,0.165058,0.226613,0.309813,0.536426
2,456781,354.333333,318.333333,32.000000,90.333333,66.000000,18.333333,0.333333,5.666667,32.333333,...,0.666667,75.333333,0.666667,0.333333,6.333333,0.0,0.284039,0.350249,0.397572,0.747821
3,457705,522.666667,451.000000,62.333333,108.666667,71.333333,20.666667,0.333333,16.333333,54.000000,...,1.000000,119.000000,7.333333,3.666667,9.000000,0.0,0.241815,0.340703,0.397539,0.738241
4,457759,565.666667,495.333333,68.666667,134.333333,88.333333,30.333333,0.000000,15.666667,77.333333,...,1.000000,98.000000,2.333333,1.000000,13.000000,0.0,0.270820,0.349677,0.425280,0.774957


In [36]:
dfFilteredLabelsSorted.head()

,playerId,rbis,homeRuns,runs,onBasePercentage,stolenBases
0,445276,0,0,0,0.000000,0
1,455117,12,4,11,0.245161,0
2,456781,21,3,10,0.290503,0
3,457705,57,13,51,0.333333,1
4,457759,18,3,14,0.287958,2


In [37]:
dfFilteredTrainSorted.pop("playerId")
playerFeatures = np.array(dfFilteredTrainSorted)

In [38]:
playerFeatures

array([[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [3.11000000e+02, 2.80333333e+02, 2.73333333e+01, ...,
        2.26612986e-01, 3.09813371e-01, 5.36426358e-01],
       [3.54333333e+02, 3.18333333e+02, 3.20000000e+01, ...,
        3.50248917e-01, 3.97572054e-01, 7.47820970e-01],
       ...,
       [1.03000000e+02, 9.20000000e+01, 1.10000000e+01, ...,
        3.13725490e-01, 3.15217391e-01, 6.28942882e-01],
       [5.00500000e+02, 4.57500000e+02, 5.80000000e+01, ...,
        3.43549840e-01, 4.30204546e-01, 7.73754386e-01],
       [1.58000000e+02, 1.45000000e+02, 1.50000000e+01, ...,
        3.10126582e-01, 3.31034483e-01, 6.41161065e-01]], shape=(597, 23))

In [39]:
normalize = layers.Normalization()

In [40]:
normalize.adapt(playerFeatures)

In [41]:
dfFilteredLabelsSorted.pop("playerId")
playerLabels = np.array(dfFilteredLabelsSorted)

In [42]:
playerLabels

array([[ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [12.        ,  4.        , 11.        ,  0.24516129,  0.        ],
       [21.        ,  3.        , 10.        ,  0.29050279,  0.        ],
       ...,
       [63.        , 13.        , 62.        ,  0.35508637,  5.        ],
       [26.        ,  4.        , 16.        ,  0.30731707,  3.        ],
       [55.        ,  8.        , 73.        ,  0.3273906 , 10.        ]],
      shape=(597, 5))

In [43]:
totalFeatureLen = len(playerFeatures)
eightyPercent = math.floor(0.8 * totalFeatureLen)
tenPercent    = math.floor(0.1 * totalFeatureLen)

playerFeaturesTrain   = playerFeatures[:eightyPercent]
playerFeaturesVal     = playerFeatures[eightyPercent + 1:eightyPercent + tenPercent]
playerFeaturesTest    = playerFeatures[eightyPercent + tenPercent + 1:]

playerLabelsTrain = playerLabels[:eightyPercent]
playerLabelsVal   = playerLabels[eightyPercent + 1:eightyPercent + tenPercent]
playerLabelsTest  = playerLabels[eightyPercent + tenPercent + 1:]

In [44]:
print(len(playerFeatures))
print(len(playerFeaturesTrain))
print(len(playerFeaturesVal))
print(len(playerFeaturesTest))
print(len(playerFeaturesTrain) + len(playerFeaturesTest) + len(playerFeaturesTest))

print("-----------------------------------")

print(len(playerLabels))
print(len(playerLabelsTrain))
print(len(playerLabelsVal))
print(len(playerLabelsTest))
print(len(playerLabelsTrain) + len(playerLabelsVal) + len(playerLabelsTest))

597
477
58
60
597
-----------------------------------
597
477
58
60
595


In [45]:
normPlayerModel = tf.keras.Sequential([
    normalize,
    layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.5),
    layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.5),
    layers.Dense(5)
])

In [62]:
normPlayerModel.compile(loss = tf.keras.losses.MeanSquaredError(), optimizer = tf.keras.optimizers.Adam( weight_decay=0.0001))

normPlayerModel.fit(playerFeaturesTrain, playerLabelsTrain, epochs=50, batch_size=128, validation_data=(playerFeaturesVal, playerLabelsVal))

Epoch 1/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 110ms/step - loss: 206.4538 - val_loss: 261.3826
Epoch 2/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - loss: 198.6848 - val_loss: 259.3226
Epoch 3/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 204.7958 - val_loss: 256.7762
Epoch 4/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - loss: 191.5362 - val_loss: 255.4686
Epoch 5/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - loss: 201.7728 - val_loss: 255.0842
Epoch 6/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - loss: 210.4492 - val_loss: 255.2709
Epoch 7/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - loss: 203.1659 - val_loss: 255.7451
Epoch 8/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - loss: 223.6368 - val_loss: 256.9728
Epoch 9/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - loss: 206.0485 - val_loss: 258.1736
Epoch 10/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - loss: 199.3813 - val_loss: 259.4483
Epoch 11/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - loss: 193.5622 - val_loss: 260.0265
Epoch 12/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 

In [63]:
normPlayerModel.evaluate(playerFeaturesTest, playerLabelsTest, batch_size=12)

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 407.6911


407.6910705566406